# Reproducibility & Environments

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 2/6

A result you cannot re-create is a rumour. This lesson makes training runs *replayable*: controlled randomness, pinned dependencies, config-driven scripts, tidy run folders and versioned data.

## 🎯 Learning Objectives

- Name the six usual suspects behind irreproducible results
- Pin dependencies with `requirements.txt` and snapshot an environment with `pip freeze`
- Seed **every** random source (Python, NumPy, scikit-learn) with one `set_all_seeds` utility and *prove* it works
- Drive training from a `dataclass` config saved next to the model as JSON
- Organise experiments into `runs/{timestamp}_{name}/` folders holding config, metrics and model
- Explain why git alone fails for datasets and how DVC-style content addressing fixes it

## 1. The Usual Suspects: Where Reproducibility Dies

You rerun last month's notebook and get a different model. Nobody hacked you — one of these quietly moved:

| Suspect | How it bites | First defence |
|---|---|---|
| Dependency drift | A teammate installs `numpy 2.x`, your script silently changes behaviour | Pinned versions (Section 2) |
| Data mutation | Someone edits `loans_v1.csv` in place — same filename, new dataset | Immutable raw copies + hashing (Section 7) |
| Uncontrolled randomness | Splits, shuffles, initialisations all draw from RNGs | Seeds everywhere (Section 4) |
| Hardware / threading | Different CPU count changes floating-point summation order | Tolerance-aware tests, recorded hardware |
| Hidden state | A global variable, a mutated module-level list, a half-run notebook kernel | Fresh process per run, configs over globals |
| Undocumented choices | "I think we dropped outliers that day?" | Config file + run log (Sections 5–6) |

**Example:** the cheapest demonstration — one unseeded line, two different universes.

In [ ]:
# Same code executed twice -> two different train/test splits
import numpy as np
from sklearn.model_selection import train_test_split

X = np.arange(200).reshape(-1, 1)

first_five = lambda split: split[1][:5].ravel()

a = first_five(train_test_split(X, test_size=0.2, shuffle=True))
b = first_five(train_test_split(X, test_size=0.2, shuffle=True))

print("call #1 test rows:", a)
print("call #2 test rows:", b)
print("Identical?", np.array_equal(a, b), "<- the split drew fresh randomness each time")

# Fix: seed the global NumPy RNG before EACH call
np.random.seed(42)
c = first_five(train_test_split(X, test_size=0.2, shuffle=True))
np.random.seed(42)
d = first_five(train_test_split(X, test_size=0.2, shuffle=True))
print("\nseeded call #1:", c)
print("seeded call #2:", d)
print("Identical?", np.array_equal(c, d))

## 2. Pinning Dependencies

`import numpy` resolves to whatever sits in `site-packages` *today*. Two machines, two months apart, two subtly different libraries — and suddenly "it works on my machine" is a bug report.

**Syntax:** exact pins vs ranges in `requirements.txt`:

```bash
# requirements.txt  --  application: pin EXACT versions
numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0
joblib==1.6.0

# Ranges are for LIBRARIES that other people install alongside theirs:
# numpy>=2.0,<3.0      <- flexible within known-good bounds
```

**Workflow:** develop freely, then freeze what actually worked:

```bash
pip freeze > requirements.lock      # EVERY package + exact version (full closure)
pip install -r requirements.lock    # teammate rebuilds your exact environment

python -m venv .venv                # isolated interpreter per project (see Module 05)
source .venv/bin/activate           # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

Modern projects move metadata into `pyproject.toml`, with a lockfile produced by tools like `uv pip compile` or `pip-tools`:

```toml
# pyproject.toml
[project]
name = "churn-trainer"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = [
    "scikit-learn==1.9.0",
    "pandas==3.0.3",
]
```

**Offline equivalent:** you do not need `pip freeze` to *see* the idea — Python can report its own loaded versions:

In [ ]:
# Snapshot the key packages of THIS environment (mini pip freeze)
from importlib.metadata import version

for pkg in ["numpy", "pandas", "scikit-learn", "matplotlib", "joblib"]:
    try:
        print(f"{pkg}=={version(pkg)}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

print("\nPaste these lines next to your experiment and 'works on my machine'")
print("becomes 'works on the machine this file describes'.")

## 3. Virtualenvs in One Breath

Full virtualenv mechanics live in Module 05 — here is the MLOps-sized version:

```bash
python -m venv .venv                  # create a private interpreter + site-packages
source .venv/bin/activate             # macOS/Linux   (.venv\Scripts\activate on Windows)
pip install -r requirements.txt       # install INTO the venv, not the system
deactivate                            # leave
```

Rule: **one project, one venv, one lockfile.** If two projects share an interpreter, they share their bugs.

## 4. Set All Seeds (and Prove It)

Randomness enters ML through many doors: `random.shuffle`, NumPy draws, `train_test_split`, forest bootstraps, weight init. Close every door with **one utility**, then *verify* by running the identical training twice.

> 🔍 **Under the Hood:** Python deliberately scrambles string hashes on every interpreter start (*hash randomisation*) to defend against hash-flooding attacks. Side effect: the iteration order of a `set` of strings differs between processes, which can flip which duplicate column wins, which tie breaks first, even feature order in dict-backed paths. The fix, `PYTHONHASHSEED`, is read **once, at interpreter start-up** — setting `os.environ["PYTHONHASHSEED"]` inside a running process documents intent but cannot un-scramble the already-built hash secret. Export it before launching: `PYTHONHASHSEED=42 python train.py`.

**Syntax:** general form of the utility:

```python
import os, random
import numpy as np

def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)  # honoured by the NEXT process, not this one
    random.seed(seed)                         # Python's built-in RNG
    np.random.seed(seed)                      # legacy global NumPy RNG (sklearn reads this)
    rng = np.random.default_rng(seed)         # modern Generator API for YOUR code
    return rng
```

In [ ]:
import os
import random

import numpy as np


def set_all_seeds(seed: int = 42) -> np.random.Generator:
    """Seed every random source the classic stack touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)   # takes effect for child processes
    random.seed(seed)                          # built-in `random` module
    np.random.seed(seed)                       # global RNG used by sklearn's default paths
    return np.random.default_rng(seed)         # hand THIS to your own sampling code


rng = set_all_seeds(7)
print("Seeded. Sample draw:", rng.integers(0, 100, size=5))

In [ ]:
# PROOF OF REPLAYABILITY: identical training, twice, bit-for-bit predictions
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)


def train_once(seed: int):
    set_all_seeds(seed)                                   # reset ALL generators
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                              shuffle=True, random_state=seed)
    model = RandomForestClassifier(n_estimators=50, random_state=seed)  # propagate!
    model.fit(X_tr, y_tr)
    return model.predict(X_te)


pred_a = train_once(42)
pred_b = train_once(42)
same = (pred_a == pred_b).all()
print(f"Run 1 == Run 2 ?  {same}")
assert same, "predictions diverged -- a random door is still open!"
print("Reproducibility verified: two cold starts, byte-identical predictions.")
print("\nKey habit: pass `random_state=seed` INTO estimators too -- seeds reset the")
print("generators, estimators still need to be told to use them.")

## 5. Config-Driven Training

Hyperparameters typed into a cell die with the cell. Put them in a **config object** — then save the config beside the model, and any run becomes a `(code, config, data)` triple you can replay.

**Syntax:** a `dataclass` gives you typed defaults plus instant JSON serialisation via `asdict`. For CLI entry points, let `argparse` point at a config file instead of 15 flags:

```python
# train.py
import argparse, json

parser = argparse.ArgumentParser()
parser.add_argument("--config", default="config.json")
args = parser.parse_args()          # python train.py --config experiments/v3.json
cfg = TrainConfig(**json.load(open(args.config)))
```

In [ ]:
import json
from dataclasses import asdict, dataclass
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)


@dataclass
class TrainConfig:
    """Everything a training run needs, in one typed bundle."""
    model_type: str = "logreg"
    lr: float = 0.01            # kept for SGD-style models / future use
    epochs: int = 30
    c: float = 1.0              # inverse regularisation strength (LogisticRegression)
    seed: int = 42
    dataset_tag: str = "breast_cancer@2026-08-26"


cfg = TrainConfig(lr=0.05, c=0.5, epochs=10)
print(cfg)

# --- save ---
cfg_path = Path("sample_data") / "config.json"
cfg_path.write_text(json.dumps(asdict(cfg), indent=2))

# --- load (fresh "process") ---
restored = TrainConfig(**json.loads(cfg_path.read_text()))

print(cfg_path.read_text())
assert restored == cfg, "round trip changed the config!"
print("JSON round trip survived: restored == original ->", restored == cfg)

## 6. The Experiment Directory Convention

Once runs multiply, `model.pkl` and `final_final_v2.pkl` become archaeology. Give every run its own folder — named by timestamp and purpose — holding everything needed to explain and replay it:

```text
runs/
  20260826_0930_baseline/
      config.json     <- exact hyperparameters (the dataclass above)
      metrics.json    <- accuracy, recall, anything you will quote later
      model.joblib    <- the fitted artefact itself
  20260826_1147_calibrated/
      ...
```

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import joblib


def make_run_dir(base: str = "sample_data/runs", name: str = "run",
                 timestamp: str | None = None) -> Path:
    """runs/{timestamp}_{name}/ -- pass timestamp=None in real life for 'now'."""
    root = Path(base)
    ts = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = root / f"{ts}_{name}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def save_artifacts(run_dir: Path, config: dict, metrics: dict, model=None) -> Path:
    run_dir = Path(run_dir)
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    if model is not None:
        joblib.dump(model, run_dir / "model.joblib")
    return run_dir


# ---- a complete, self-describing training run -------------------------------
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

set_all_seeds(cfg.seed)
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=cfg.seed, stratify=y)
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=cfg.c, max_iter=2000))
pipe.fit(X_tr, y_tr)

run_dir = make_run_dir(name="baseline", timestamp="20260826_0930")  # fixed stamp = deterministic lesson
save_artifacts(run_dir, asdict(cfg),
               metrics={"test_accuracy": round(pipe.score(X_te, y_te), 4)},
               model=pipe)

print("files in run dir:", sorted(p.name for p in run_dir.iterdir()))
print((run_dir / "metrics.json").read_text())

In [ ]:
# Replay check: a COLD process loads folder contents and reproduces the model
import json
from pathlib import Path

import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

run_dir = Path("sample_data/runs/20260826_0930_baseline")
saved_cfg = json.loads((run_dir / "config.json").read_text())
revived = joblib.load(run_dir / "model.joblib")

X, y = load_breast_cancer(return_X_y=True)
_, X_te, _, y_te = train_test_split(X, y, test_size=0.25,
                                    random_state=saved_cfg["seed"], stratify=y)

original = pipe.predict(X_te)
assert (revived.predict(X_te) == original).all()
print(f"Loaded '{run_dir.name}' (C={saved_cfg['c']}, seed={saved_cfg['seed']})")
print("Reloaded model reproduces the original predictions exactly.")
print("\nThat folder IS the experiment: anyone with the repo can re-run or audit it.")

## 7. Data Versioning: Why Git Is Not Enough

Git stores *line diffs of text*. Your data breaks those assumptions:

- a 2 GB binary file diff is meaningless noise;
- one edited row changes nothing a human can review;
- GitHub blocks files over 100 MB outright;
- models need the *exact bytes*, not "roughly Tuesday's version".

**DVC** (Data Version Control) keeps the bytes in cheap storage and puts tiny `.dvc` pointer files in git — the pointer records a content hash, so identical data always maps to the same version ID.

```bash
pip install dvc
dvc init                              # creates .dvc/ alongside .git/

dvc add data/raw/loans.csv            # hash the file -> write loans.csv.dvc pointer
                                      # AND auto-add data/raw/loans.csv to .gitignore
git add data/raw/loans.csv.dvc data/raw/.gitignore
git commit -m "loans v1"              # git tracks the POINTER, storage holds the BYTES

dvc remote add -d store s3://team-bucket/dvcstore
dvc push                              # upload this snapshot for the team
dvc pull                              # teammate downloads EXACTLY this version
git checkout v2-tag && dvc checkout   # old tag -> old data, matched by hash
```

The git/DVC dance in one line: **git answers "which code ran?", DVC answers "on which bytes?"**

**Offline equivalent:** the heart of DVC is just *content addressing* — hash the file, treat the hash as its identity. You can feel that right now:

In [ ]:
import hashlib
from pathlib import Path

data_path = Path("sample_data") / "data.csv"
data_path.write_text("user_id,amount\n101,250\n102,90\n103,310\n")


def content_id(path) -> str:
    """DVC-style identity: a hash of the file's exact bytes."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]


id_v1 = content_id(data_path)
print("dataset id v1:", id_v1)

with data_path.open("a") as f:          # one row sneaks in overnight...
    f.write("104,175\n")

id_v2 = content_id(data_path)
print("dataset id v2:", id_v2)

assert id_v1 != id_v2
print("Filename never changed, but the CONTENT ID moved -> pipelines can now detect it.")
print("(This is exactly how `dvc status` knows your data is out of date.)")

## 8. Sidebar: The Limits of Determinism

Even fully seeded, perfectly pinned code has honest limits — know them so you don't chase ghosts:

- **GPUs:** parallel float reductions and cuDNN auto-tuning make some deep-learning ops non-deterministic unless you force deterministic algorithms (often at a speed cost).
- **Threads & BLAS:** matrix products sum in thread-dependent order; results agree to ~1e-16 but not bitwise. Compare with tolerances, not `==`.
- **Even plain CPUs bite:** floating-point addition is not associative.

```python
(0.1 + 0.2) + 0.3   !=   0.1 + (0.2 + 0.3)    # True: they really differ
```

Practical rule: chase *bitwise* equality for classic ML on CPU; chase *metric-level* equality (same confusion matrix, same loss to 6 decimals) anywhere else.

In [ ]:
# Feel the float fuzz yourself
a, b, c = 0.1, 0.2, 0.3

left = (a + b) + c
right = a + (b + c)

print(f"(0.1+0.2)+0.3 = {left!r}")
print(f"0.1+(0.2+0.3) = {right!r}")
print("bitwise equal:", left == right, "| difference:", abs(left - right))

print("\nMoral: assert with np.allclose(...), never with == on floats.")

## 9. The Reproducibility Checklist

Pin this above your desk. A run is "reproducible" when every box ticks:

| # | Question | Evidence to leave behind |
|---|---|---|
| 1 | Which code? | git commit SHA recorded in the run log |
| 2 | Which environment? | `requirements.lock` (or frozen versions) beside the run |
| 3 | Which data? | Dataset content-hash / DVC tag in the config |
| 4 | Which randomness? | One master seed + `random_state=` on every estimator |
| 5 | Which settings? | Saved `config.json`, never retyped values |
| 6 | Which artefacts? | `runs/{ts}_{name}/` with model + metrics |
| 7 | Does a cold rerun match? | Assert predictions/metrics match within tolerance |

If question 7 fails, questions 1–6 tell you where to look — that is the whole game.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Loose pins (`numpy>=1.20`) in an app's requirements | Transitive deps still drift between machines | Exact pins or a generated lockfile for apps; ranges only for libraries |
| Setting `PYTHONHASHSEED` inside the script | Hash secret was built at interpreter start — too late | Export it before launch: `PYTHONHASHSEED=42 python train.py` |
| Seeding NumPy but not `random` (or vice versa) | Some library path still rolls dice | Route everything through one `set_all_seeds` |
| Editing the raw CSV "just this once" | The true input is gone; every past result is now unauditable | Raw layer is read-only; transformations write derived copies |
| Loading a joblib pickle under a different sklearn version | Warning today, wrong behaviour or crash tomorrow | Store library versions beside every model; rebuild before loading |
| Run dirs named only by timestamp | Two runs in the same second overwrite each other | `{timestamp}_{name}` (+ short uuid if paranoid) |

## 💡 Best Practices & Pro Tips

- **One seed constant, passed everywhere** — defined in the config, handed to splitters and estimators alike. Grep for `random_state` during code review.
- **Never mutate what you read.** `df = pd.read_csv(raw)` then transform *into new objects/files*; the raw file outlives every experiment.
- **Save the config WITH the model, always.** A checkpoint without hyperparameters is a sculpture without a blueprint.
- **Prefer `default_rng(seed)` for your own sampling** — the modern NumPy Generator is cleaner than the global legacy stream you are seeding for sklearn's benefit.
- **AI-engineering relevance:** nothing here is ML-specific. Prompt templates, agent tool lists and temperature settings deserve the same treatment — version them like configs, hash them like data, and your LLM evaluations become replayable too.
- **Make "rerun from scratch" someone's actual job** in CI: a nightly smoke run that replays yesterday's config catches rot while it is cheap.

## 📌 Summary

| Tool / Pattern | What it does | Example |
|---|---|---|
| `requirements.txt` pins | Freeze the dependency universe | `scikit-learn==1.9.0` |
| `pip freeze` | Snapshot the whole environment | `pip freeze > requirements.lock` |
| `set_all_seeds(k)` | Reset Python + NumPy randomness | `set_all_seeds(cfg.seed)` |
| `random_state=k` | Make one estimator deterministic | `RandomForestClassifier(random_state=seed)` |
| `@dataclass` + `asdict` | Typed config, instant JSON | `json.dumps(asdict(cfg))` |
| `make_run_dir` / `save_artifacts` | Self-describing run folders | `runs/20260826_0930_baseline/model.joblib` |
| `hashlib.sha256(bytes)` | Content-address a dataset | Same bytes ⇒ same ID (DVC's core trick) |
| `dvc add / push / pull` | Git tracks pointers, storage tracks bytes | `git checkout v2 && dvc checkout` |

Key takeaways:

- Irreproducibility is never mysterious — it is one of six moving parts you did not pin down.
- Seeds reset generators; estimators still need `random_state` told explicitly.
- A run is replayable when `(code, environment, data, config)` are all recoverable — save them together.
- Verify reproducibility with an assertion, not a vibe: rerun and compare predictions.

## 🔗 Next Lesson

Up next: **[03_Experiment_Tracking_MLflow](../03_Experiment_Tracking_MLflow/notes.ipynb)** — fifty runs later, which config won? Tracking systems (MLflow, plus a from-scratch file tracker you can run anywhere).